In [ ]:
import scipy.signal.signaltools

def _centered(arr, newsize):
    # Return the center newsize portion of the array.
    newsize = np.asarray(newsize)
    currsize = np.array(arr.shape)
    startind = (currsize - newsize) // 2
    endind = startind + newsize
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

scipy.signal.signaltools._centered = _centered

import eventstudy as es
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import pandas_datareader as web

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfMinute.pkl")

In [ ]:
df_major = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/MinuteIntersection1stdev.pkl")
aa = [] 
for i in df_major.values:
    aa.append(pd.to_Minutetime(i))
bb = pd.DataFrame(data=aa)
bb.index = bb.values
bb.rename(columns={0:"Minute"},inplace=True)
bb.index = bb["Minute"]
bb.rename(columns={"Minute":"aa"},inplace=True)
df = df[["VIX return","Sentiment return"]]
df.rename(columns={"Minute":"aa"},inplace=True)
df_es = bb.merge(df, on='Minute')
df_es = df_es.drop(columns={"aa"})
df_es["Minute"] = df_es.index

# df = df_es

In [ ]:
df["Minute"] = df.index
df.dropna(inplace=True)
df = df[(df.T != 0).any()]
df = df.loc[df["Sentiment return"]!=0]
df = df.loc[df["VIX return"]!=0]
df_Minutes = df

In [ ]:
events_db = []
for Minute, row in df_es.T.iteritems():
    print(Minute)
    new_row = {'event_Minute': np.Minutetime64(row["Minute"]),'market_ticker':'Sentiment return','security_ticker': 'VIX return'}#row["Company"]}
    events_db.append(new_row)

In [ ]:
events_db

In [ ]:
returns = df_Minutes[["Sentiment return","VIX return"]]


In [ ]:
len(df_Minutes)

In [ ]:
len(events_db)

In [ ]:
returns.index.names = ['Minute']

In [ ]:
returns

In [ ]:
returns.to_csv("../CSV/returns_list.csv")

In [ ]:
returns.sort_index(inplace=True)

In [ ]:
returns

In [ ]:
es.Single.import_returns("../CSV/returns_list.csv")
# es.Single.import_FamaFrench("../CSV/FamaFrenchSPY.csv")

In [ ]:
tickers={"VIX return"}

In [ ]:
events=events_db

In [ ]:
# tickers = ['SPY']
releases = dict()
ticker= "VIX return"
events = [event for event in events_db if event['security_ticker']==ticker]
print(ticker)
releases[ticker] = es.Multiple.from_list(
    events,
    es.Single.market_model,
    event_window= (-5,+5)
)

In [ ]:
releases

In [ ]:
for ticker, event in releases.items():
    event.plot()
    print(ticker)
    # plt.figure(figsize=(20, 20))
    plt.xlabel('Event Day')
    plt.title('Event Study - Daily Frequency\nSentiment Return and VIX Return')
# plt.show()

In [ ]:
aa = pd.DataFrame(data=event.tstat)